In [27]:
#load package
#install.packages("e1071", repos = "https://cloud.r-project.org")
library(e1071)

#load raw data
train<- read.csv("classification_train.csv", stringsAsFactors = FALSE)
test<- read.csv("classification_test.csv", stringsAsFactors = FALSE)

#convert target to 5-class factor
y <- factor(train$alwaysAnxious, levels = c(-2, -1, 0, 1, 2))

#only use numeric predictors
num_cols <- setdiff(names(train)[sapply(train, is.numeric)], "alwaysAnxious")
x_train <- as.matrix(train[, num_cols, drop = FALSE])
x_test  <- as.matrix(test[, num_cols, drop = FALSE])

#standardise train data predictors
center_val <- colMeans(x_train)
scale_val <- apply(x_train,2,sd)
#avoid division by 0
scale_val[scale_val== 0] <- 1

#scale train and test data with the same mean and sd
x_train_s <- scale(x_train,center = center_val, scale =scale_val)
x_test_s <- scale(x_test,center = center_val, scale =scale_val)

#set weights for 5-class target
class_table <- table(y)
class_weights <- as.numeric(sum(class_table)/(length(class_table)*class_table))
names(class_weights) <- names(class_table)

# model1 0.52704

In [43]:
#define macro-F1 function
macro_f1 <- function(truth, pred) {
  f1 <- sapply(levels(truth), function(k) {
    tp<-sum(pred == k & truth == k)
    fp<-sum(pred == k & truth != k)
    fn<-sum(pred != k & truth == k)
    #calculate precision and recall for each class
    p <- if (tp + fp == 0) 0 else tp / (tp + fp)
    r <- if (tp + fn == 0) 0 else tp / (tp + fn)
    #calculate F1 score for each class
    if (p + r == 0) 0 else 2 * p * r / (p + r)
  })
      mean(f1)}

#split data into 5-folds
set.seed(1)
fold <- integer(length(y))
#assign samples from each class evenly into 5 folds
for (lev in levels(y)) {
  idx <- sample(which(y == lev))
  fold[idx] <- rep(1:5, length.out = length(idx))}

#run 5-folds CV
scores <- numeric(5)
for (f in 1:5) {
#training fold
  tr <- fold != f
#validation fold
  va <- fold == f
#train weighted RBF SVM on the training folds with chosen parameters(model1)
  m <- svm(x = x_train_s[tr, ], y = y[tr],kernel = "radial", cost=2, gamma=0.05,class.weights = class_weights, scale=FALSE)
    #predict validation fold and calculate macro-F1
  scores[f] <- macro_f1(y[va], predict(m, x_train_s[va, ]))
}

cat("Macro-F1 for each fold:",scores,"\n")
cat("Mean 5-fold macro-F1:",mean(scores),"\n")

Macro-F1 for each fold: 0.34933 0.3892346 0.409286 0.3856296 0.4206096 
Mean 5-fold macro-F1: 0.390818 


In [48]:
#model1: kaggle public score: 0.52704
set.seed(1)
fin.mod1 <- svm(x = x_train_s, y = y,kernel = "radial", cost=2, gamma=0.05,class.weights = class_weights, scale = FALSE)

pred.label <- predict(fin.mod1, x_test_s)
pred.label <- as.integer(as.character(pred.label))

write.csv(
  data.frame("RowIndex" = seq_along(pred.label), "Prediction" = pred.label),
  "ClassificationPredictLabel_model1.csv",
  row.names = FALSE
)

# model2 0.59364

In [51]:
#define macro-F1 function
macro_f1 <- function(truth, pred) {
  f1 <- sapply(levels(truth), function(k) {
    tp<-sum(pred == k & truth == k)
    fp<-sum(pred == k & truth != k)
    fn<-sum(pred != k & truth == k)
    #calculate precision and recall for each class
    p <- if (tp + fp == 0) 0 else tp / (tp + fp)
    r <- if (tp + fn == 0) 0 else tp / (tp + fn)
    #calculate F1 score for each class
    if (p + r == 0) 0 else 2 * p * r / (p + r)
  })
      mean(f1)}

#split data into 5-folds
set.seed(1)
fold <- integer(length(y))
#assign samples from each class evenly into 5 folds
for (lev in levels(y)) {
  idx <- sample(which(y == lev))
  fold[idx] <- rep(1:5, length.out = length(idx))}

#run 5-folds CV
scores <- numeric(5)
for (f in 1:5) {
#training fold
  tr <- fold != f
#validation fold
  va <- fold == f
#train weighted RBF SVM on the training folds with chosen parameters(model2)
  m <- svm(x = x_train_s[tr, ], y = y[tr],kernel = "radial", cost=1.75, gamma=0.045,class.weights = class_weights, scale=FALSE)
    #predict validation fold and calculate macro-F1
  scores[f] <- macro_f1(y[va], predict(m, x_train_s[va, ]))
}

cat("Macro-F1 for each fold:",scores,"\n")
cat("Mean 5-fold macro-F1:",mean(scores),"\n")

Macro-F1 for each fold: 0.3375165 0.39271 0.4111579 0.4031403 0.3942099 
Mean 5-fold macro-F1: 0.3877469 


In [52]:
#model2: kaggle public score: 0.59364
set.seed(2)
fin.mod2 <- svm(x = x_train_s, y = y,kernel = "radial", cost=1.75, gamma=0.045,class.weights = class_weights, scale = FALSE)

pred.label <- predict(fin.mod2, x_test_s)
pred.label <- as.integer(as.character(pred.label))

write.csv(
  data.frame("RowIndex" = seq_along(pred.label), "Prediction" = pred.label),
  "ClassificationPredictLabel_model2.csv",
  row.names = FALSE
)

Model 1 and Model 2 had very similar local CV performance. Model 1 achieved a 5-fold CV macro-F1 of about 0.3908, while Model 2 achieved about 0.3877, so the difference was only around 0.003. This small gap does not provide strong evidence that Model 2 performs worse locally. I also checked nearby RBF SVM settings using repeated cross-validation, and their mean CV macro-F1 scores were very close: 0.4118, 0.4080, and 0.4072. This suggests that Model 2 was not clearly overfitting compared with the nearby alternatives.

Although Model 2 had a much higher Kaggle public score, I do not assume that this public score will exactly represent the final private score. The public leaderboard only evaluates part of the test set, so it may overestimate the final performance. However, since Model 2 had comparable CV performance and the strongest public leaderboard result, I selected it as the final weighted RBF SVM model.


In [58]:
#final selected model: Model 2
set.seed(2)
fin.mod <- svm(x = x_train_s, y = y,kernel = "radial", cost=1.75, gamma=0.045,class.weights = class_weights, scale = FALSE)

pred.label <- predict(fin.mod, x_test_s)
pred.label <- as.integer(as.character(pred.label))

write.csv(
  data.frame("RowIndex" = seq_along(pred.label), "Prediction" = pred.label),
  "ClassificationPredictLabel.csv",
  row.names = FALSE
)